# Reddit Posts Preprocessing

Preprocesses the phase-organized Reddit dataset (`00-data/reddit-data/<phase>/<subreddit>/*.csv`) into a single cleaned CSV per crisis phase: threads get their post **title**, every row carries its **subreddit**, comments expose an anonymized **reply_to** flag, and all author identifiers (`author`, `author_fullname`, `u/` and `@` mentions) are removed.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import re
import csv
import emoji
import pathlib
import nltk
from nltk.tokenize import sent_tokenize
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# --- papermill parameters (injected by runner) ---
crisis_name = "ateneo-drowning"  # default; overridden by papermill
phase_name = "02-confirmation"  # default; overridden by papermill

### Cross-Phase Comment Distribution (EDA)

Compute a raw distribution of threads and comments per phase across the whole reorganized dataset before processing any single phase. The table and bar chart are read from the raw files, so they reflect the full corpus.

In [ ]:
# --- Distribution of comments per phase (raw data) ---
data_dir = f'../../00-data/{crisis_name}/reddit-data'
phase_dirs = sorted(d for d in os.listdir(data_dir)
                    if os.path.isdir(os.path.join(data_dir, d)))

dist_rows = []
for ph in phase_dirs:
    files = glob.glob(os.path.join(data_dir, ph, '*', '*.csv'))
    n_posts, n_comments = 0, 0
    for f in files:
        with open(f, newline='', encoding='utf-8') as fh:
            for row in csv.DictReader(fh):
                t = row.get('type')
                if t == 'post':
                    n_posts += 1
                elif t == 'comment':
                    n_comments += 1
    dist_rows.append({'phase': ph, 'threads': n_posts, 'comments': n_comments})

dist_df = pd.DataFrame(dist_rows)
dist_df['share'] = dist_df['comments'] / dist_df['comments'].sum() * 100
dist_df['avg_comments_per_thread'] = dist_df['comments'] / dist_df['threads'].replace(0, np.nan)
display(dist_df.round(1))

In [ ]:
# --- Bar chart of comments per phase ---
phase_order = ['00-unrelated', '01-onset', '02-confirmation',
               '03-spread', '04-reckoning', '05-lingering']
plot_df = dist_df.set_index('phase').reindex(phase_order)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#9e9e9e'] + ['#4c72b0'] * 9   # grey = unrelated, blue = crisis phases
ax.bar(plot_df.index, plot_df['comments'], color=colors, alpha=0.85)
for x, v in enumerate(plot_df['comments']):
    ax.text(x, v, f"{int(v):,}", ha='center', va='bottom', fontsize=8)
ax.set_title('Distribution of Reddit Comments per Phase')
ax.set_ylabel('Comments')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()

fig_dir = pathlib.Path('../../outputs/figures')
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / 'reddit-comments-per-phase.png', bbox_inches='tight')
plt.show()

### Batch Dataset Loading and Thread Enrichment

Each file in the phase folder is a single thread: one `post` row followed by its `comment` rows. The post's `text` starts with the thread title (first line), so the title is extracted once per file and attached to every row of that thread.

In [ ]:
# 1. Load the selected phase directory
base_data_dir = f'../../00-data/{crisis_name}/reddit-data'

phase_path = os.path.join(base_data_dir, phase_name)
file_list = sorted(glob.glob(os.path.join(phase_path, '*', '*.csv')))

if not file_list:
    print(f"Warning: No CSV files found in '{phase_path}'")
else:
    frames = []
    for f in file_list:
        df = pd.read_csv(f)
        post = df[df['type'] == 'post']
        if post.empty:
            continue
        thread_id = str(post.iloc[0]['id']).replace('t3_', '')
        title = str(post.iloc[0]['text']).split('\n', 1)[0]
        df['thread_id'] = thread_id
        df['title'] = title
        frames.append(df)

    combined_df = pd.concat(frames, ignore_index=True)
    print(f"--- Dataset Summary for {phase_name} ---")
    print(f"Total files merged: {len(file_list)}")
    print(f"Total rows and columns: {combined_df.shape}\n")

    print("--- Column Data Types & Non-Null Counts ---")
    combined_df.info()

    print("\n--- Statistical Summary ---")
    print(combined_df.describe(include="all"))

### Reply Structure and Author Removal

Comments expose `replying_to_user` (the parent comment's handle). This is converted to an anonymized `reply_to` marker plus an `is_reply` flag, and all username-bearing columns (`author`, `author_fullname`, `replying_to_user`, raw `metadata_json`) are dropped.

### Auto-Generated and Bot Comment Removal

Remove rows from AutoModerator (karma-gate and flair-check notices), `sneakpeekbot` (link previews), and image-only placeholder comments (`<image>`). These carry no linguistic signal for sentiment or topic analysis.

In [ ]:
# Drop auto-generated / bot / image-placeholder rows
auto_bot_mask = (
    combined_df['author'].isin(['AutoModerator', 'sneakpeekbot'])
    | (combined_df['text'] == '<image>')
)
before = len(combined_df)
combined_df = combined_df[~auto_bot_mask].copy()
print(f"Removed {before - len(combined_df)} auto-generated / bot / image-placeholder rows ({len(combined_df):,} remaining)")

# Quick verification
for name in ['AutoModerator', 'sneakpeekbot']:
    n = (combined_df['author'] == name).sum()
    if n:
        print(f"  WARNING: {n} {name} rows still present")
n_img = (combined_df['text'] == '<image>').sum()
if n_img:
    print(f"  WARNING: {n_img} <image> rows still present")

In [ ]:
# Capture usernames before removal (used later to verify no leaks)
all_authors = set(combined_df['author'].fillna(''))

# is_reply + redacted reply_to
combined_df['is_reply'] = (combined_df['type'] == 'comment').astype(int)
combined_df['reply_to'] = np.where(
    (combined_df['type'] == 'comment')
    & combined_df['replying_to_user'].fillna('').ne(''),
    '[redacted]', '')

# drop author-identifying columns
combined_df = combined_df.drop(columns=['author', 'author_fullname', 'replying_to_user', 'metadata_json'])

print("Author-identifying columns removed.")
combined_df[['id', 'subreddit', 'title', 'reply_to', 'is_reply']].head()

### Mention Anonymization

Replace `u/username` and `@handle` mentions in the text with a `[redacted]` token so no user handle survives in the corpus.

In [ ]:
def redact_mentions(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'(?<![\w/])u/[\w\-\.]+', '[redacted]', text, flags=re.IGNORECASE)
    text = re.sub(r'(?<!\w)@[\w\-\.]+', '[redacted]', text)
    return text

combined_df['text'] = combined_df['text'].apply(redact_mentions)

sample_redact = combined_df['text'][combined_df['text'].str.contains('[redacted]', na=False)]
print(f"Rows containing a redacted mention: {len(sample_redact)}")

### Text Cleaning Pipeline

Same cleaning as the YouTube notebook: lowercase, emoji demojization, URL removal, character-repetition normalization, and an `is_shouting` flag for ALL-CAPS text.

In [ ]:
def clean_text_for_crisis(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = emoji.demojize(text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

combined_df['is_shouting'] = combined_df['text'].apply(
    lambda x: 1 if isinstance(x, str) and x.isupper() and len(x) > 5 else 0)
combined_df['clean_text'] = combined_df['text'].apply(clean_text_for_crisis)

### Preprocessing Validation

Side-by-side raw vs cleaned text, plus emoji and shouting checks, before dropping the original column.

In [ ]:
sample = combined_df.sample(min(10, len(combined_df)), random_state=42)[['text', 'clean_text', 'is_shouting']]
display(sample)

emoji_mask = combined_df['text'].str.contains(r'[\U0001F622\U0001F621\U0001F62D\U0001F602\U0001F44D\u2764\uFE0F]', na=False)
print(f"Comments with emojis: {emoji_mask.sum()}")

### Sentence Segmentation

Long posts/comments are split into individual sentences via NLTK's `sent_tokenize` for downstream per-sentence transformer scoring (512-token limit).

In [ ]:
nltk.download('punkt_tab', quiet=True)

combined_df['sentences'] = combined_df['clean_text'].apply(sent_tokenize)

long_mask = combined_df['clean_text'].str.len() > 500
print(f"Rows exceeding ~500 chars: {long_mask.sum()} ({long_mask.mean():.1%})")
combined_df.head()

### Per-Batch Summary

Descriptive stats for this phase only — total rows, threads, subreddit split, date range, shouting and emoji rates (mirrors the YouTube notebook's export report).

In [ ]:
combined_df['created_time'] = pd.to_datetime(combined_df['created_time'])

print("--- Preprocessed Summary ---")
print(f"Total rows: {len(combined_df)}")
print(f"Threads: {combined_df['thread_id'].nunique()}")
print(f"Subreddits:\n{combined_df['subreddit'].value_counts().to_string()}")
print(f"Date range: {combined_df['created_time'].min()} to {combined_df['created_time'].max()}")
print(f"Shouting (%): {combined_df['is_shouting'].mean() * 100:.1f}%")
print(f"With emojis: {emoji_mask.sum()} ({emoji_mask.mean() * 100:.1f}%)")

### Deduplication and Export

Drop comments with identical cleaned text, then export the phase as a single CSV under `output/reddit/`.

In [ ]:
before_dedup = len(combined_df)
combined_df = combined_df.drop_duplicates(subset='clean_text', keep='first')
print(f"Dropped {before_dedup - len(combined_df)} duplicate texts")

output_dir = pathlib.Path(f'../output/{crisis_name}/reddit')
output_dir.mkdir(parents=True, exist_ok=True)

out_df = combined_df.drop(columns=['text']).rename(columns={'clean_text': 'text'})
out_df = out_df[['id', 'subreddit', 'title', 'reply_to', 'is_reply', 'score',
                 'created_time', 'text', 'is_shouting', 'sentences']]
csv_path = output_dir / f'{phase_name}.csv'
out_df.to_csv(csv_path, index=False)

print(f"Exported: {csv_path.resolve()}")

### Anonymization Validation

Confirm no `u/` or `@` handles survive in the output text and that no author username from this phase appears verbatim in the corpus.

In [ ]:
residual = out_df['text'].str.contains(r'(?<![\w/])u/|(?<!\w)@', regex=True, na=False).sum()
print(f"Rows with residual handles in text: {residual}")

authors_list = sorted(a for a in all_authors if a)

def count_leaks(text):
    tl = text.lower()
    return sum(a.lower() in tl for a in authors_list if a)

leaks = out_df['text'].apply(count_leaks).sum()
print(f"Verbatim username leaks: {leaks}")